# K513 · Week 7, Session 1
## Sign it off, or send it back

All semester, an analyst has been doing the same work you did: reading a column, drawing a chart, tidying a table, splitting the data, building and evaluating models. Before any of it goes to a client, someone has to review it. Today that is your team.

The notebook below has **12 checks**, one or more from every week of the course. In each one the analyst runs some code and writes a conclusion. **Every cell runs without an error, and every conclusion has a problem.** Your job, for each check:

1. Find what is wrong.
2. Fix it, in a new code cell under the analyst's code if necessary. If you can directly answer what's wrong, why, and how to correct it without coding, that is fine. AI code completion can be turned on and you may use it. But you need to write out the question for AI to code.
3. Write one line for your team's report: what is wrong, and the fix.

**Split the checks across your team** (for example, checks 1–3, 4–6, 7–9, 10–12), then explain each one to the rest of the team before you submit. The report asks *why* the conclusion is wrong, and that is the part your team has to agree on.

**Submit:** one report per team, in Canvas — *Week 7 Bug Hunt - Team Report*.
One student can be responsible for summarizing all the bug checks and submitting the summary to Canvas.

Run the two setup cells first. After that, each check runs on its own.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import mean_squared_error, confusion_matrix, roc_auc_score

pd.set_option('display.precision', 3)
plt.rcParams['figure.figsize'] = (5, 4)
SEED = 42

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/"

boston_df  = pd.read_csv(DATA_URL + "BostonHousing.csv")
bike_df    = pd.read_csv(DATA_URL + "bikeshare.csv")
machine_df = pd.read_csv(DATA_URL + "Machine%20Failure%20Data.csv")
loan_df    = pd.read_csv(DATA_URL + "loan.csv")

# The customers and orders from Week 2, Session 2
customer_df = pd.DataFrame({
    'CustomerID': [1111, 2222, 3333, 4444, 5555, 6666, 7777, 8888, 9999],
    'NAME': ['Charles', 'Bertram', 'Barbara', 'Bala', 'Dennis',
             'Davis', 'Scott', 'John', 'Stephen'],
})
orders_2015 = pd.DataFrame({
    'OrderID':    [1020, 1021, 1022, 1023, 1024],
    'CustomerID': [1111, 1111, 2222, 3333, 3333],
})
orders_2016 = pd.DataFrame({
    'OrderID':    [1825, 1826, 1827],
    'CustomerID': [4444, 5555, 6666],
})
order_df = pd.concat([orders_2015, orders_2016], ignore_index=True)

print("boston_df: ", boston_df.shape)
print("bike_df:   ", bike_df.shape)
print("machine_df:", machine_df.shape)
print("loan_df:   ", loan_df.shape)
print("customer_df", customer_df.shape, " order_df", order_df.shape)

# Week 1 · Reading one variable

### Check 1 · Boston — which columns are extreme?

In [ ]:
print(boston_df.skew().sort_values(ascending=False).head(3))

sns.boxplot(data=boston_df, y='CHAS')
plt.show()

**The analyst's conclusion:** "`CHAS` is the second most skewed column in the data (3.41), and its boxplot shows a group of outlier tracts far above the box. Outliers like these will distort any model, so I removed every tract with `CHAS` = 1."

# Week 2 · Drawing it

### Check 2 · Bikeshare — which day is busiest?

In [ ]:
# weekday: 0 = Sunday, 1 = Monday, ..., 6 = Saturday
sns.barplot(data=bike_df, x='weekday', y='num_shared', errorbar=None)
plt.show()

**The analyst's conclusion:** "Friday (5) is clearly our busiest day, about 460 rides a day ahead of Sunday. Move extra bikes into the stations every Friday."

### Check 3 · Boston — what is the relationship between LSTAT and home value?

In [ ]:
cols = ['MEDV', 'RM', 'LSTAT', 'CRIM']
sns.heatmap(data=boston_df[cols].corr(), annot=True)
plt.show()

**The analyst's conclusion:** "`LSTAT` has the strongest correlation with `MEDV` (−0.74). So the relationship is a straight line: every extra point of `LSTAT` lowers home value by the same amount, whether `LSTAT` is 5 or 30."

# Week 2 · Putting a table into shape

### Check 4 · Bikeshare — total rides by weather

In [ ]:
bike_df.pivot_table(index='weather', values='num_shared')

**The analyst's conclusion:** "Total rides over the two years: clear days 4,877, cloudy days 4,036, wet days 1,803. Clear days carry only a little more of our business than cloudy days."

### Check 5 · Customers and orders — how many orders does a customer place?

In [ ]:
merged = customer_df.merge(order_df, on='CustomerID')
orders_per_customer = merged.groupby('CustomerID')['OrderID'].count()

print("average orders per customer:", round(orders_per_customer.mean(), 2))

**The analyst's conclusion:** "Our customers place 1.33 orders each, on average."

# Week 3 · Machine learning concepts

### Check 6 · Machines — how accurate is the machine failure model?

In [ ]:
X = machine_df.drop(columns=['Failure?', 'Product ID'])
y = (machine_df['Failure?'] == 'Yes').astype(int)

categorical_features = ['Type']
continuous_features = ['Air Temp', 'Proc Temp', 'Rtn Speed', 'Torque', 'Tool Wear']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)

tree_pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('cont', 'passthrough', continuous_features)])

tree_model = Pipeline([('preprocessor', tree_pre),
                       ('classifier', DecisionTreeClassifier(max_depth=4,
                                                             random_state=SEED))])
tree_model.fit(X_train, y_train)

print(f"accuracy: {tree_model.score(X, y):.3f}")

**The analyst's conclusion:** "The model is 92.0% accurate, so we can expect it to be right about 92% of the machines on the floor next month."

### Check 7 · Boston — how well does the pricing model do?

In [ ]:
X = boston_df.drop(columns=['CHAS'])
y = boston_df['MEDV']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=SEED)

house_model = LinearRegression().fit(X_train, y_train)

print(f"R² on the test set: {house_model.score(X_test, y_test):.3f}")

**The analyst's conclusion:** "R² of 1.000 on tracts the model never saw. The model is perfect, and we can price every tract exactly."

# Week 4 · Regression

### Check 8 · Boston — is the model good enough to use?

In [ ]:
X = boston_df.drop(columns=['MEDV'])
y = boston_df['MEDV']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=SEED)
house_model = LinearRegression().fit(X_train, y_train)

rmse = np.sqrt(mean_squared_error(y_test, house_model.predict(X_test)))
print(f"test RMSE:            ${rmse * 1000:,.0f}")
print(f"average tract value:  ${y_test.mean() * 1000:,.0f}")

**The analyst's conclusion:** "A typical miss of \$4,523 on tracts worth about \$21,400 on average is a 21% error. Regression does not work on this data; tell the client to keep pricing tracts by hand."

### Check 9 · Loans — does location matter?

In [ ]:
X = loan_df[['Income', 'CDAccount', 'ZIP Code']]
y = loan_df['PersonalLoan']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)

preprocessor = ColumnTransformer(transformers=[
    ('cont', StandardScaler(), ['Income', 'ZIP Code']),
    ('bin', 'passthrough', ['CDAccount'])])

loan_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=SEED))]).fit(X_train, y_train)

coefficients = pd.Series(loan_model.named_steps['classifier'].coef_[0],
                         index=['Income', 'ZIP Code', 'CDAccount'])
print(coefficients.round(3))

**The analyst's conclusion:** "The `ZIP Code` coefficient is almost zero (−0.024), so where a customer lives makes no difference to whether they accept a loan. Marketing can ignore location."

# Week 5 · Classification

### Check 10 · Machines — does Product ID help?

In [ ]:
X = machine_df.drop(columns=['Failure?'])
y = (machine_df['Failure?'] == 'Yes').astype(int)

categorical_features = ['Type', 'Product ID']
continuous_features = ['Air Temp', 'Proc Temp', 'Rtn Speed', 'Torque', 'Tool Wear']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)

preprocessor = ColumnTransformer(transformers=[
    ('cont', StandardScaler(), continuous_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=SEED))])
model.fit(X_train, y_train)

print('training accuracy:', round(model.score(X_train, y_train), 3))
print('    test accuracy:', round(model.score(X_test, y_test), 3))

**The analyst's conclusion:** "Without `Product ID`, this model's training accuracy was 0.820. Adding `Product ID` raised it to 0.888. Keep it in the model."

# Week 6 · Evaluating a classifier

Checks 11–12 use the bank's loan campaign from Week 6. Run this cell first. It builds the same three models as Week 6, Session 1.

In [ ]:
X = loan_df[['Income', 'CDAccount']]
y = loan_df['PersonalLoan']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)

preprocessor = ColumnTransformer(transformers=[
    ('cont', StandardScaler(), ['Income']),
    ('bin', 'passthrough', ['CDAccount'])])

baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
logreg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=SEED))]).fit(X_train, y_train)
tree = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_train, y_train)

BUDGET       = 1250   # dollars approved for the campaign, on our 1,250 test customers
PER_MAILING  = 10     # what one mailing costs
PER_ACCEPTOR = 500    # what one accepted loan is worth

print(f"{len(X_test)} test customers, {y_test.sum()} of them acceptors")

### Check 11 · Loans — which model should we use?

In [ ]:
for name, m in [('Baseline      ', baseline),
                ('Logistic      ', logreg),
                ('Decision tree ', tree)]:
    print(f"{name} accuracy {m.score(X_test, y_test):.4f}")

**The analyst's conclusion:** "All three models are more than 90% accurate. The decision tree is the most accurate, at 0.9184, so we mail the customers the tree picks."

### Check 12 · Loans — how many customers should we mail?

In [ ]:
mail = logreg.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, mail).ravel()
mailed = tp + fp
cost = PER_MAILING * mailed

print(f"mailed     {mailed}")
print(f"spent      ${cost:,} of ${BUDGET:,}")
print(f"acceptors  {tp}")
print(f"profit     ${PER_ACCEPTOR * tp - cost:,}")

**The analyst's conclusion:** "We mailed 73 customers for \$730, reached 44 acceptors and made \$21,270, and we came in \$520 under budget. Return the unused \$520 to finance."

# Your team's report

Submit **one report per team** in Canvas: *Week 7 Bug Hunt - Team Report*. Paste this into the text box and fill in one line per check. At the end of class, add the last line and submit:

```
Team number, and everyone on the team who is here today:
Check 1  — what is wrong:            — the fix:
Check 2  — what is wrong:            — the fix:
...
Check 12 — what is wrong:            — the fix:
The one thing our team still has to fix before Sunday:
```

"What is wrong" should name the mistake in a sentence a teammate would understand, for example "the average was taken after the merge dropped the customers with no orders", not "merge problem".